# 3. The pipeline: gene selection and the graph model under nested cross-validation

Question: does a graph model on genes selected inside each training part predict overall survival better than the clinical covariates alone?

What enters: the cohort table of notebook 1 and the STRING edge lists of notebook 2.

What leaves: the fold files, the comparison rows, the frozen final model and the tables in `results/03_pipeline/`; the first run stays in its `first_run/` folder.

Result: rendered under the imports from the saved summary.

How to run: the nbconvert command of notebook 1; seconds in read mode, hours in compute mode.

In [ ]:
import hashlib
import json
import subprocess
import sys
import time
import warnings
from importlib.metadata import version
from pathlib import Path

warnings.simplefilter("ignore", FutureWarning)  # torch's import-time notice about torch.jit
warnings.filterwarnings("ignore", module="tqdm")  # tqdm asks for a widget the notebook lacks

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sksurv.linear_model import CoxnetSurvivalAnalysis, CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored, concordance_index_ipcw
from sksurv.util import Surv
from torch_geometric.data import Batch, Data
from torch_geometric.nn import SAGEConv, global_mean_pool
from torchsurv.loss.cox import neg_partial_log_likelihood

In [ ]:
summary_file = Path("../results/03_pipeline/summary.csv")
if summary_file.exists():
    saved = pd.read_csv(summary_file, index_col=0)["value"]
    display(Markdown(f"Result: over {saved['outer folds done']} of 5 outer folds the graph model "
                     f"has a mean Harrell's C of {float(saved['graph model mean C']):.3f}, the "
                     f"clinical-only Cox {float(saved['clinical-only Cox mean C']):.3f}; the "
                     f"paired difference is {float(saved['graph minus clinical mean']):.3f} with "
                     "interval "
                     f"{float(saved['graph minus clinical lower']):.3f} to "
                     f"{float(saved['graph minus clinical upper']):.3f}."))
else:
    display(Markdown("Result: not yet run."))

## Settings

Every grid, seed and cut-off of the procedure is set here once. The compute switch is set to no, so the notebook reads the saved results; setting it to yes computes the outer folds and comparison rows whose file is missing, then the final model.

In [ ]:
COMPUTE = False             # no: read the saved results; yes: compute what has no saved file
SCORE_METABRIC = False      # no until the supervisors confirm the procedure; the model is frozen
RUN_NO_GRAPH = True         # the same model with the edges removed, after the mandatory results
RUN_COMPARISON_ROWS = True  # the rows of the section "Comparison rows"
IDENTITY_SIZE = 16          # numbers per gene in the learnable identity of three comparison rows
NEIGHBOURS_KEPT = 5         # strongest STRING neighbours per selected gene in a possible next graph
DENSER_EDGE_FILE = Path("../results/02_gene_graph/edges_400.csv.gz")  # a denser graph
RESULTS = Path("../results/03_pipeline")
DATA_FILE = Path("../data/processed/tcga.parquet")
EDGE_FILE = Path("../results/02_gene_graph/edges_700.csv.gz")
OUTER_FOLDS = 5             # the outer loop scores the procedure
INNER_FOLDS = 5             # the inner loop chooses every setting
FOLD_SEED = 0               # fixed fold assignment
TORCH_SEED = 0              # fixed weight initialisation per training
EXPRESSED_AT = 1.0          # a gene counts as expressed at log2(TPM + 1) >= 1
EXPRESSED_IN = 0.2          # in at least 20% of the training patients
L1_RATIOS = [0.5, 0.75, 1.0]  # elastic net between half L1 and the LASSO
PENALTIES = 25              # penalties on each training part's own path
SMALLEST_PENALTY = 0.05     # each path ends at 5% of its own largest penalty
LIST_MIN, LIST_MAX = 10, 1000  # eligible: 10 to 1,000 genes kept in every inner fit part
HIDDEN_SIZES = [64, 128]    # width of the hidden layers of every network
DROPOUTS = [0.2, 0.5]       # dropout after each hidden layer
LEARNING_RATE = 0.001       # Adam
WEIGHT_DECAY = 0.0001       # L2 penalty of Adam
EPOCHS = 40                 # fixed; the last epoch is scored, no epoch is picked on validation data
CLINICAL_RIDGE = 0.001      # a small ridge keeps the clinical Cox finite when a level is rare
CLINICAL_SHARE = 0.01       # clinical columns beside the genes: 1% of their penalty, kept finite
HORIZON_DAYS = 5 * 365.25   # Uno's C is computed up to five years

## Data

The cohort table is the one notebook 1 wrote, checked by its hash, with the outcome, five clinical covariates and one column per pool gene. The edge lists are the STRING graphs of notebook 2 at score 700 and, for two comparison rows, 400.

In [ ]:
table = pd.read_parquet(DATA_FILE)
cohort = json.loads(Path("../results/01_data/cohort.json").read_text())["tcga"]
data_hash = hashlib.sha256(DATA_FILE.read_bytes()).hexdigest()
assert data_hash == cohort["file sha256"], "the cohort table is not the one notebook 1 wrote"
OUTCOME = ["os_days", "os_event"]
CLINICAL = ["age", "stage", "er", "pr", "her2"]
CATEGORIES = CLINICAL[1:]
GENES = [column for column in table.columns if column.startswith("ENSG")]
assert len(GENES) + len(CLINICAL) + len(OUTCOME) == table.shape[1], "an unexpected column"
edges = pd.read_csv(EDGE_FILE)
edge_hash = hashlib.sha256(EDGE_FILE.read_bytes()).hexdigest()
denser_edges = pd.read_csv(DENSER_EDGE_FILE)
display(Markdown(f"{len(table)} patients, {int(table['os_event'].sum())} deaths, {len(GENES)} pool "
                 f"genes, {len(edges)} STRING edges at score 700 or more and {len(denser_edges)} "
                 "at 400 or more."))

## Which patients each step sees

The outer loop holds one fifth of the patients out and scores the procedure on it; inside each training part the inner loop holds one fifth out twice, once to choose the selector's settings and once to choose the graph model's settings. The figure counts patients and deaths in each part for the first outer fold.

In [ ]:
def folds(part, count):
    """The (training, held-out) row pairs of one part, stratified by the event flag."""
    splitter = StratifiedKFold(count, shuffle=True, random_state=FOLD_SEED)
    return list(splitter.split(part.index, part["os_event"]))


def outcome_of(part):
    """The outcome of one part in the library's structured form."""
    return Surv.from_arrays(part["os_event"].astype(bool), part["os_days"])


def part_label(name, part):
    """Patients and deaths of one part, for the figure."""
    return f"{name}: {len(part)} patients, {int(part['os_event'].sum())} deaths"


outer = folds(table, OUTER_FOLDS)
first_training = table.iloc[outer[0][0]]
bars = [("outer fold 1 training part", "outer fold 1 test part", table, outer[0])]
for number, inner_rows in enumerate(folds(first_training, INNER_FOLDS), start=1):
    bars.append((f"inner fold {number} fit part", f"inner fold {number} validation part",
                 first_training, inner_rows))

In [ ]:
figure, axis = plt.subplots(figsize=(9, 5))
for position, (fit_name, score_name, part, (fit_rows, score_rows)) in enumerate(bars):
    fit_part, score_part = part.iloc[fit_rows], part.iloc[score_rows]
    axis.barh(-position, len(fit_part), color="tab:blue")
    axis.barh(-position, len(score_part), left=len(fit_part), color="tab:orange")
    axis.text(5, -position, part_label(fit_name, fit_part), va="center", color="white",
              fontsize=8.5)
    axis.text(len(part) + 10, -position, part_label(score_name, score_part), va="center",
              fontsize=8.5)
axis.set_xlim(0, len(table) * 1.6)  # room for the labels of the held-out parts
axis.set_yticks([])
axis.set_xlabel("Patients")
axis.set_title("Inner folds are drawn from the outer training part; stage A (selector settings) "
               "and stage B (graph settings) use the same five inner folds")
plt.show()

## Preprocessing inside the training part

The expression filter and the z-scoring of genes and age are fitted on the training part of each fold, inner or outer, and applied to its held-out part. The categorical covariates are one-hot encoded with their unknown level; the Cox models drop the most common level of each, learned on the training part.

In [ ]:
def scaled_expression(train, other):
    """Keep genes expressed in enough training patients and z-score them on the training part."""
    expressed = (train[GENES] >= EXPRESSED_AT).mean() >= EXPRESSED_IN
    kept = list(expressed.index[expressed])
    scaler = StandardScaler().fit(train[kept])
    train_scaled = pd.DataFrame(scaler.transform(train[kept]), index=train.index, columns=kept)
    other_scaled = pd.DataFrame(scaler.transform(other[kept]), index=other.index, columns=kept)
    return train_scaled, other_scaled


def clinical_vectors(train, other, drop=None):
    """Age z-scored and stage, ER, PR and HER2 one-hot, fitted on the training part."""
    encoder = OneHotEncoder(drop=drop, sparse_output=False, handle_unknown="ignore")
    encoder.fit(train[CATEGORIES])
    scaler = StandardScaler().fit(train[["age"]])
    vectors = []
    for part in (train, other):
        age = scaler.transform(part[["age"]])
        vectors.append(np.hstack([age, encoder.transform(part[CATEGORIES])]))
    return vectors


def reference_levels(train):
    """The most common level of each category in a training part, dropped by the Cox models."""
    return [train[name].mode()[0] for name in CATEGORIES]

## Scores

Harrell's C is the share of patient pairs whose predicted risks are in the same order as their survival times, among the pairs the data can order; 0.5 is chance. Uno's C reweights the pairs for censoring and is read up to five years.

In [ ]:
def harrell(part, risk):
    """Harrell's C-index of a risk score on one part."""
    return concordance_index_censored(part["os_event"].astype(bool), part["os_days"], risk)[0]


def uno(train, part, risk):
    """Uno's C-index up to the horizon, with censoring weights from the training part."""
    return concordance_index_ipcw(outcome_of(train), outcome_of(part), risk, tau=HORIZON_DAYS)[0]


def both_scores(train, test, risk):
    """Harrell's C and Uno's C of one risk score on a test part."""
    scores = {"Harrell's C": float(harrell(test, risk)),
              "Uno's C (5 years)": float(uno(train, test, risk))}
    return scores

## Stage A: the selector's settings

The selector is an elastic-net Cox model: a Cox model whose penalty sets most gene coefficients to zero, so the genes left are the selected list. Each inner fit part computes its own path of 25 penalties per L1 ratio, from the penalty that keeps no gene down to 5% of it; its validation part only scores them. A candidate is an L1 ratio and a place on the path, eligible when every inner fit part keeps 10 to 1,000 genes there; the highest mean score wins, ties going to the stronger penalty.

In [ ]:
def selector_inputs(train, other, with_clinical):
    """Selector features of a training and a held-out part, and the names of the gene columns."""
    x_train, x_other = scaled_expression(train, other)
    train_features = x_train.to_numpy()
    other_features = x_other.to_numpy()
    if with_clinical:
        c_train, c_other = clinical_vectors(train, other, reference_levels(train))
        train_features = np.hstack([train_features, c_train])
        other_features = np.hstack([other_features, c_other])
    return train_features, other_features, list(x_train.columns)


def selector_path(features, outcome, l1_ratio, gene_count):
    """The library's path on one training part, its penalties set by the gene columns alone."""
    genes_only = CoxnetSurvivalAnalysis(l1_ratio=l1_ratio, n_alphas=PENALTIES,
                                        alpha_min_ratio=SMALLEST_PENALTY)
    genes_only.fit(features[:, :gene_count], outcome)
    assert len(genes_only.alphas_) == PENALTIES, "the library returned a shorter path"
    if features.shape[1] == gene_count:
        return genes_only
    factor = np.ones(features.shape[1])
    factor[gene_count:] = CLINICAL_SHARE  # the clinical columns after the genes
    model = CoxnetSurvivalAnalysis(l1_ratio=l1_ratio, alphas=genes_only.alphas_,
                                   penalty_factor=factor)
    return model.fit(features, outcome)


def inner_candidates(fit_part, score_part, l1_ratio, with_clinical):
    """One inner fold: its own path, the genes kept at each place and the validation C-index."""
    fit_features, score_features, genes = selector_inputs(fit_part, score_part, with_clinical)
    model = selector_path(fit_features, outcome_of(fit_part), l1_ratio, len(genes))
    counts = (model.coef_[:len(genes)] != 0).sum(axis=0)
    rows = []
    for place, penalty in enumerate(model.alphas_, start=1):
        risk = model.predict(score_features, alpha=penalty)
        row = {"place": place, "penalty": float(penalty), "genes": int(counts[place - 1]),
               "inner C": float(harrell(score_part, risk))}
        rows.append(row)
    return rows

In [ ]:
def stage_a(train, inner, with_clinical=False):
    """Choose the L1 ratio and the place on the path by the mean inner validation C-index."""
    rows = []
    for l1_ratio in L1_RATIOS:
        for fold, (fit_rows, score_rows) in enumerate(inner, start=1):
            fit_part, score_part = train.iloc[fit_rows], train.iloc[score_rows]
            for row in inner_candidates(fit_part, score_part, l1_ratio, with_clinical):
                row["l1 ratio"] = l1_ratio
                row["inner fold"] = fold
                rows.append(row)
    per_fold = pd.DataFrame(rows)
    grid = per_fold.groupby(["l1 ratio", "place"], as_index=False).agg(
        mean_c=("inner C", "mean"), fewest=("genes", "min"), most=("genes", "max"))
    grid.columns = ["l1 ratio", "place", "inner mean C", "fewest genes", "most genes"]
    grid["eligible"] = (grid["fewest genes"] >= LIST_MIN) & (grid["most genes"] <= LIST_MAX)
    assert grid["eligible"].any(), "no setting keeps 10 to 1,000 genes in every inner fit part"
    ranked = grid[grid["eligible"]].sort_values(["inner mean C", "place"], ascending=[False, True])
    return ranked.iloc[0], grid, per_fold


def select_genes(train, other, setting, with_clinical=False):
    """Refit the selector on a training part's own path; its genes and risk at the chosen place."""
    train_features, other_features, genes = selector_inputs(train, other, with_clinical)
    model = selector_path(train_features, outcome_of(train), setting["l1 ratio"], len(genes))
    index = int(setting["place"]) - 1  # places count from 1, the strongest penalty
    nonzero = model.coef_[:len(genes), index] != 0
    risk = model.predict(other_features, alpha=model.alphas_[index])
    return list(pd.Index(genes)[nonzero]), risk, float(model.alphas_[index])

## The graph model

Each patient is one graph: the selected genes are its nodes, the patient's z-scored expression of a gene is the node's feature, and STRING edges at score 700 or more connect the nodes. Two GraphSAGE layers pass information between neighbours, mean pooling makes one vector per patient, the clinical vector is appended and a two-layer head gives one log-hazard, trained with the Cox partial likelihood for a fixed number of epochs.

In [ ]:
class GraphModel(torch.nn.Module):
    """Two GraphSAGE layers, mean pooling, the clinical vector and a two-layer head."""

    def __init__(self, hidden, dropout, clinical_size, gene_count=0, identity_size=0):
        """Layers sized by the width, the dropout, the clinical vector and the optional identity."""
        super().__init__()
        self.identity = None
        if identity_size:
            self.identity = torch.nn.Embedding(gene_count, identity_size)
        self.layer_1 = SAGEConv(1 + identity_size, hidden)
        self.layer_2 = SAGEConv(hidden, hidden)
        self.dropout = torch.nn.Dropout(dropout)
        self.head = torch.nn.Sequential(torch.nn.Linear(hidden + clinical_size, hidden),
                                        torch.nn.ReLU(), torch.nn.Dropout(dropout),
                                        torch.nn.Linear(hidden, 1))

    def forward(self, graphs, clinical):
        """One log-hazard per patient graph."""
        features = graphs.x
        if self.identity is not None:
            gene_position = torch.arange(len(features)) % self.identity.num_embeddings
            features = torch.cat([features, self.identity(gene_position)], dim=1)
        hidden = self.dropout(torch.relu(self.layer_1(features, graphs.edge_index)))
        hidden = self.dropout(torch.relu(self.layer_2(hidden, graphs.edge_index)))
        pooled = global_mean_pool(hidden, graphs.batch)
        return self.head(torch.cat([pooled, clinical], dim=1)).squeeze(1)

## The plain networks

Three comparison rows use a multilayer perceptron (MLP), a network of fully connected layers that reads the selected genes as one vector, so each input position is one known gene. Its gene part has two layers of the graph model's width and dropout, and the same head takes their output and the clinical vector. Without genes the head reads the clinical vector alone; without clinical columns, the gene part alone.

In [ ]:
class PlainModel(torch.nn.Module):
    """Two fully connected layers on the genes, then a two-layer head with the clinical vector."""

    def __init__(self, hidden, dropout, gene_count, clinical_size):
        """Layers sized by the width, the dropout, the number of genes and the clinical vector."""
        super().__init__()
        self.genes = None
        head_size = clinical_size
        if gene_count:
            self.genes = torch.nn.Sequential(torch.nn.Linear(gene_count, hidden), torch.nn.ReLU(),
                                             torch.nn.Dropout(dropout),
                                             torch.nn.Linear(hidden, hidden), torch.nn.ReLU(),
                                             torch.nn.Dropout(dropout))
            head_size = head_size + hidden
        self.head = torch.nn.Sequential(torch.nn.Linear(head_size, hidden), torch.nn.ReLU(),
                                        torch.nn.Dropout(dropout), torch.nn.Linear(hidden, 1))

    def forward(self, genes, clinical):
        """One log-hazard per patient."""
        if self.genes is None:
            return self.head(clinical).squeeze(1)
        return self.head(torch.cat([self.genes(genes), clinical], dim=1)).squeeze(1)

## Inputs, training and prediction

A network's inputs are the patient graphs or the gene matrix, then the clinical vectors. Every network starts from the same seed and is trained the same way: Adam, the Cox partial likelihood on all training patients at once, a fixed number of epochs.

In [ ]:
def edges_among(genes, edge_list):
    """The rows of an edge list whose two genes are both in the given list."""
    return edge_list[edge_list["gene_a"].isin(genes) & edge_list["gene_b"].isin(genes)]


def patient_graphs(expression, genes, edge_table):
    """One graph per patient on the given genes, with the edges of the table between them."""
    position = pd.Series(range(len(genes)), index=genes)
    kept = edges_among(genes, edge_table)
    ends = np.vstack([position[kept["gene_a"]].to_numpy(), position[kept["gene_b"]].to_numpy()])
    edge_index = torch.tensor(np.hstack([ends, ends[::-1]]), dtype=torch.long)
    values = torch.tensor(expression[genes].to_numpy(), dtype=torch.float32)
    graphs = [Data(x=row.unsqueeze(1), edge_index=edge_index) for row in values]
    return Batch.from_data_list(graphs)


def model_inputs(variant, expression, genes, clinical):
    """A network's inputs: patient graphs or a gene matrix, then the clinical vectors."""
    clinical = torch.tensor(clinical, dtype=torch.float32)
    if variant["kind"] == "graph":
        return patient_graphs(expression, genes, variant["edges"]), clinical
    gene_matrix = torch.tensor(expression[genes].to_numpy(), dtype=torch.float32)
    if not variant["clinical"]:
        clinical = clinical[:, :0]  # no clinical column
    return gene_matrix, clinical

In [ ]:
def new_model(variant, inputs, hidden, dropout):
    """An untrained network of the variant's kind, its weights started from the fixed seed."""
    torch.manual_seed(TORCH_SEED)
    first, clinical = inputs
    if variant["kind"] == "graph":
        gene_count = first.x.shape[0] // first.num_graphs  # every patient graph has the same nodes
        return GraphModel(hidden, dropout, clinical.shape[1], gene_count, variant["identity"])
    return PlainModel(hidden, dropout, first.shape[1], clinical.shape[1])


def train_network(model, inputs, part):
    """Train a network for the fixed number of epochs with the Breslow Cox loss, full batch."""
    optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    event = torch.tensor(part["os_event"].to_numpy(), dtype=torch.bool)
    days = torch.tensor(part["os_days"].to_numpy(), dtype=torch.float32)
    model.train()
    for epoch in range(EPOCHS):
        optimiser.zero_grad()
        loss = neg_partial_log_likelihood(model(*inputs), event, days, ties_method="breslow")
        loss.backward()
        optimiser.step()
    return model


def log_hazard(model, inputs):
    """The trained network's log-hazard per patient, with dropout off."""
    model.eval()
    with torch.no_grad():
        return model(*inputs).numpy()

## Stage B: the graph model's settings

On the same five inner folds, the selector is refitted at the chosen setting on the inner fit part alone, its genes form the graph, one model per width and dropout is trained on the fit part and scored on the validation part, and the setting with the highest mean score wins. The same functions tune the networks of the comparison rows. An inner fit part that keeps fewer than two genes scores 0.5 for every setting and is counted.

In [ ]:
def inner_network_scores(fit_part, score_part, setting, variant):
    """Validation C-index of each width and dropout on one inner fold, genes selected inside it."""
    genes = []
    if variant["genes"]:
        genes, _, _ = select_genes(fit_part, score_part, setting)
    scores = {}
    if variant["genes"] and len(genes) < 2:
        for hidden in HIDDEN_SIZES:
            for dropout in DROPOUTS:
                scores[(hidden, dropout)] = 0.5
        return scores, len(genes)
    x_fit, x_score = scaled_expression(fit_part, score_part)
    c_fit, c_score = clinical_vectors(fit_part, score_part)
    fit_inputs = model_inputs(variant, x_fit, genes, c_fit)
    score_inputs = model_inputs(variant, x_score, genes, c_score)
    for hidden in HIDDEN_SIZES:
        for dropout in DROPOUTS:
            untrained = new_model(variant, fit_inputs, hidden, dropout)
            model = train_network(untrained, fit_inputs, fit_part)
            scores[(hidden, dropout)] = harrell(score_part, log_hazard(model, score_inputs))
    return scores, len(genes)


def stage_b(train, inner, setting, variant):
    """Choose the width and dropout of a network by the mean inner validation C-index."""
    totals = {}
    few_genes = 0
    for fit_rows, score_rows in inner:
        scores, gene_count = inner_network_scores(train.iloc[fit_rows], train.iloc[score_rows],
                                                  setting, variant)
        few_genes += int(variant["genes"] and gene_count < 2)
        for network_setting, score in scores.items():
            totals[network_setting] = totals.get(network_setting, 0) + score / len(inner)
    rows = []
    for (hidden, dropout), score in totals.items():
        row = {"hidden size": hidden, "dropout": dropout, "inner mean C": float(score)}
        rows.append(row)
    inner_table = pd.DataFrame(rows).sort_values("inner mean C", ascending=False)
    return inner_table.iloc[0], inner_table, few_genes

## Stage C: one model per outer fold, scored once

With both settings chosen, the selector is refitted on the whole outer training part at the chosen place on that part's own path and gives this fold's gene list; the graph model is trained on that part with the chosen setting, and the outer test part is scored once. The clinical-only Cox and the refitted selector's own risk are scored on the same split as reference rows.

In [ ]:
def clinical_cox(train, test):
    """The library's Cox model on the five covariates, the most common level of each dropped."""
    c_train, c_test = clinical_vectors(train, test, reference_levels(train))
    model = CoxPHSurvivalAnalysis(alpha=CLINICAL_RIDGE).fit(c_train, outcome_of(train))
    return model.predict(c_test)


def network_scores(train, test, genes, network_setting, variant):
    """Train one network on the training part at a chosen width and dropout; score the test part."""
    x_train, x_test = scaled_expression(train, test)
    c_train, c_test = clinical_vectors(train, test)
    train_inputs = model_inputs(variant, x_train, genes, c_train)
    test_inputs = model_inputs(variant, x_test, genes, c_test)
    hidden = int(network_setting["hidden size"])
    dropout = float(network_setting["dropout"])
    model = train_network(new_model(variant, train_inputs, hidden, dropout), train_inputs, train)
    if variant.get("identity"):
        assert model.identity.num_embeddings == len(genes), "one identity row per gene of the list"
    return both_scores(train, test, log_hazard(model, test_inputs))


def stage_c(train, test, setting, graph_setting):
    """Fit the selector, the graph model and the reference rows; score the test part once."""
    genes, selector_risk, penalty = select_genes(train, test, setting)
    scores = {"graph model": network_scores(train, test, genes, graph_setting, BASE),
              "selector as a model": both_scores(train, test, selector_risk),
              "clinical-only Cox": both_scores(train, test, clinical_cox(train, test))}
    return genes, penalty, scores

## Running the outer folds

Each outer fold runs stages A, B and C and writes its own file, so a run that stops loses one fold and a fold whose file exists is skipped. The gene list of each fold is saved beside it. The functions after the registered ones serve the comparison rows, and the last cell of this part runs whatever has no saved file.

In [ ]:
def stage_a_record(setting, grid, per_fold):
    """Stage A's choice with its grid and every inner fold's own path, as plain values."""
    record = {"l1 ratio": float(setting["l1 ratio"]), "place": int(setting["place"]),
              "inner mean C": float(setting["inner mean C"]),
              "fewest genes": int(setting["fewest genes"]),
              "most genes": int(setting["most genes"]),
              "eligible settings": int(grid["eligible"].sum()),
              "settings": grid.to_dict("records"), "inner folds": per_fold.to_dict("records")}
    return record


def stage_b_record(network_setting, inner_table, few_genes):
    """Stage B's choice with its grid, as plain values."""
    record = {"hidden size": int(network_setting["hidden size"]),
              "dropout": float(network_setting["dropout"]),
              "inner mean C": float(network_setting["inner mean C"]),
              "inner folds with fewer than 2 genes": few_genes,
              "settings": inner_table.to_dict("records")}
    return record

In [ ]:
BASE = {"kind": "graph", "genes": True, "edges": edges, "identity": 0}  # the registered model
NO_EDGES = {"kind": "graph", "genes": True, "edges": edges.iloc[:0], "identity": 0}  # no edge


def run_outer_fold(number, train_rows, test_rows):
    """Stages A, B and C of one outer fold, written to the fold's own file."""
    train, test = table.iloc[train_rows], table.iloc[test_rows]
    assert set(train.index).isdisjoint(test.index), "a patient is in both parts"
    inner = folds(train, INNER_FOLDS)
    started = time.time()
    setting, grid, per_fold = stage_a(train, inner)
    after_a = time.time()
    graph_setting, inner_table, few_genes = stage_b(train, inner, setting, BASE)
    after_b = time.time()
    genes, penalty, scores = stage_c(train, test, setting, graph_setting)
    between = edges_among(genes, edges)
    patients = {"training": len(train), "test": len(test),
                "training deaths": int(train["os_event"].sum()),
                "test deaths": int(test["os_event"].sum())}
    stage_c_record = {"genes": len(genes), "penalty": penalty, "edges": len(between),
                      "genes with an edge": len(set(between["gene_a"]) | set(between["gene_b"])),
                      "scores": scores}
    minutes = {"stage A": round((after_a - started) / 60, 1),
               "stage B": round((after_b - after_a) / 60, 1),
               "stage C": round((time.time() - after_b) / 60, 1)}
    record = {"fold": number, "patients": patients,
              "stage A": stage_a_record(setting, grid, per_fold),
              "stage B": stage_b_record(graph_setting, inner_table, few_genes),
              "stage C": stage_c_record, "minutes": minutes}
    (RESULTS / f"outer_fold_{number}.json").write_text(json.dumps(record, indent=1))
    (RESULTS / f"outer_fold_{number}_genes.txt").write_text("\n".join(genes) + "\n")

In [ ]:
def add_no_graph_row(number, train_rows, test_rows):
    """The same model with the edges removed, on the same split, list and setting."""
    record = json.loads((RESULTS / f"outer_fold_{number}.json").read_text())
    if "no-graph model" in record["stage C"]["scores"]:
        return
    train, test = table.iloc[train_rows], table.iloc[test_rows]
    genes, _, _ = select_genes(train, test, record["stage A"])
    assert len(genes) == record["stage C"]["genes"], "the refit selects another list"
    scores = network_scores(train, test, genes, record["stage B"], NO_EDGES)
    record["stage C"]["scores"]["no-graph model"] = scores
    (RESULTS / f"outer_fold_{number}.json").write_text(json.dumps(record, indent=1))

In [ ]:
IDENTITY_700 = "graph model, STRING 700, gene identity"
COMPARISON_ROWS = {
    "selector plus clinical covariates": {"kind": "cox"},
    "clinical MLP": {"kind": "plain", "genes": False, "clinical": True},
    "gene MLP": {"kind": "plain", "genes": True, "clinical": False},
    "gene plus clinical MLP": {"kind": "plain", "genes": True, "clinical": True},
    IDENTITY_700: {"kind": "graph", "genes": True, "edges": edges, "identity": IDENTITY_SIZE},
    "no-graph model, gene identity": {"kind": "graph", "genes": True, "edges": edges.iloc[:0],
                                      "identity": IDENTITY_SIZE, "settings from": IDENTITY_700},
    "graph model, STRING 400": {"kind": "graph", "genes": True, "edges": denser_edges,
                                "identity": 0},
    "graph model, STRING 400, gene identity": {"kind": "graph", "genes": True,
                                               "edges": denser_edges, "identity": IDENTITY_SIZE}}
ROW_DIR = RESULTS / "comparison_rows"


def row_file(name, number):
    """The fold file of one comparison row."""
    folder = ROW_DIR / name.replace(", ", "_").replace(" ", "_").lower()
    folder.mkdir(parents=True, exist_ok=True)
    return folder / f"outer_fold_{number}.json"


def recorded_setting(number):
    """The selector setting of one saved fold, without its scores, and the fold's record."""
    record = json.loads((RESULTS / f"outer_fold_{number}.json").read_text())
    record.pop("stage C")  # the recorded scores are never read by a comparison row
    return record["stage A"], record

In [ ]:
def selector_plus_clinical(train, test, inner):
    """Elastic-net Cox on the genes plus the clinical columns, with its own stage A."""
    setting, grid, per_fold = stage_a(train, inner, with_clinical=True)
    genes, risk, penalty = select_genes(train, test, setting, with_clinical=True)
    out = {"stage A": stage_a_record(setting, grid, per_fold), "genes": len(genes),
           "penalty": penalty, "edges": 0, "scores": both_scores(train, test, risk)}
    return out


def tuned_network(train, test, setting, variant):
    """Stage B for one network on the same inner folds, then one score on the test part."""
    network_setting, inner_table, few_genes = stage_b(train, folds(train, INNER_FOLDS), setting,
                                                      variant)
    genes = []
    if variant["genes"]:
        genes, _, _ = select_genes(train, test, setting)
    edge_list = variant.get("edges", edges.iloc[:0])
    out = {"stage B": stage_b_record(network_setting, inner_table, few_genes),
           "genes": len(genes), "edges": len(edges_among(genes, edge_list)),
           "scores": network_scores(train, test, genes, network_setting, variant)}
    return out


def network_without_tuning(variant, number, train, test, setting):
    """The control: the genes, width, dropout and seed of another row, without its edges."""
    source = json.loads(row_file(variant["settings from"], number).read_text())
    genes, _, _ = select_genes(train, test, setting)
    assert len(genes) == source["genes"], "the refit selects another list than the source row"
    out = {"settings from": variant["settings from"], "stage B": source["stage B"],
           "genes": len(genes), "edges": 0,
           "scores": network_scores(train, test, genes, source["stage B"], variant)}
    return out

In [ ]:
def run_comparison_fold(name, variant, number, train_rows, test_rows):
    """One comparison row on one outer fold, from the fold's recorded split and selector setting."""
    setting, record = recorded_setting(number)
    train, test = table.iloc[train_rows], table.iloc[test_rows]
    assert len(train) == record["patients"]["training"], "the split differs from the saved fold"
    assert set(train.index).isdisjoint(test.index), "a patient is in both parts"
    started = time.time()
    if variant["kind"] == "cox":
        out = selector_plus_clinical(train, test, folds(train, INNER_FOLDS))
    elif "settings from" in variant:
        out = network_without_tuning(variant, number, train, test, setting)
    else:
        out = tuned_network(train, test, setting, variant)
    out["fold"] = number
    out["row"] = name
    out["minutes"] = round((time.time() - started) / 60, 1)
    row_file(name, number).write_text(json.dumps(out, indent=1))

In [ ]:
RESULTS.mkdir(exist_ok=True)
if COMPUTE:
    for number, (train_rows, test_rows) in enumerate(outer, start=1):
        if not (RESULTS / f"outer_fold_{number}.json").exists():
            run_outer_fold(number, train_rows, test_rows)
    if RUN_NO_GRAPH:
        for number, (train_rows, test_rows) in enumerate(outer, start=1):
            add_no_graph_row(number, train_rows, test_rows)
    if RUN_COMPARISON_ROWS:
        for name, variant in COMPARISON_ROWS.items():
            for number, (train_rows, test_rows) in enumerate(outer, start=1):
                if not row_file(name, number).exists():
                    run_comparison_fold(name, variant, number, train_rows, test_rows)
fold_files = sorted(RESULTS.glob("outer_fold_*.json"))
fold_records = [json.loads(path.read_text()) for path in fold_files]
display(Markdown(f"{len(fold_records)} of {OUTER_FOLDS} outer folds have a saved file."))

## Stage A results

Per outer fold: the chosen L1 ratio and place on the path (1 is the strongest of the 25 penalties), the mean inner validation C-index at that setting, the fewest and most genes it kept in the five inner fit parts, the penalty at that place on the outer training part's own path, the genes it keeps there and the STRING edges among them.

In [ ]:
rows = []
for record in fold_records:
    chosen = record["stage A"]
    row = {"outer fold": record["fold"], "L1 ratio": chosen["l1 ratio"],
           "place on the path": chosen["place"], "inner mean C": round(chosen["inner mean C"], 3),
           "eligible settings of 75": chosen["eligible settings"],
           "genes in the inner fit parts": f"{chosen['fewest genes']} to {chosen['most genes']}",
           "penalty": round(record["stage C"]["penalty"], 4),
           "genes selected": record["stage C"]["genes"], "edges": record["stage C"]["edges"],
           "genes with an edge": record["stage C"]["genes with an edge"]}
    rows.append(row)
stage_a_table = pd.DataFrame(rows).set_index("outer fold")
stage_a_table.to_csv(RESULTS / "stage_a.csv")
display(Markdown(f"Stage A, {len(fold_records)} of {OUTER_FOLDS} folds."))
stage_a_table

In [ ]:
ratios = sorted(stage_a_table["L1 ratio"].unique())
offered = ", ".join(str(ratio) for ratio in L1_RATIOS)
chosen = ", ".join(str(ratio) for ratio in ratios)
ratio_text = f"the folds chose L1 ratios of {chosen} ({offered} offered)"
if ratios == [min(L1_RATIOS)]:
    ratio_text = (f"every fold chose an L1 ratio of {ratios[0]}, the lowest value the grid offers "
                  f"({offered}), so the selector sat at the edge of its grid")
display(Markdown(f"The chosen lists hold {stage_a_table['genes selected'].min()} to "
                 f"{stage_a_table['genes selected'].max()} genes; {ratio_text}."))

## Stage B results

Per outer fold: the chosen width and dropout of the graph model and its mean inner validation C-index, with the number of inner fit parts that kept fewer than two genes.

In [ ]:
rows = []
for record in fold_records:
    row = {"outer fold": record["fold"], "hidden size": record["stage B"]["hidden size"],
           "dropout": record["stage B"]["dropout"],
           "inner mean C": round(record["stage B"]["inner mean C"], 3),
           "inner fit parts with fewer than 2 genes":
               record["stage B"]["inner folds with fewer than 2 genes"]}
    rows.append(row)
stage_b_table = pd.DataFrame(rows).set_index("outer fold")
stage_b_table.to_csv(RESULTS / "stage_b.csv")
display(Markdown(f"Stage B, {len(fold_records)} of {OUTER_FOLDS} folds."))
stage_b_table

In [ ]:
graph_inner = stage_b_table["inner mean C"]
selector_inner = stage_a_table["inner mean C"]
display(Markdown(f"The inner validation C-index of the chosen graph setting ranges from "
                 f"{graph_inner.min():.3f} to {graph_inner.max():.3f}; the selector's own inner "
                 f"C-index in stage A ranges from {selector_inner.min():.3f} to "
                 f"{selector_inner.max():.3f}."))

## Results on the outer test parts

Per outer fold and model, Harrell's C and Uno's C on the outer test part, with the mean and standard deviation over the folds. The no-graph row is the same model with every edge removed.

In [ ]:
rows = []
for record in fold_records:
    for model_name, scores in record["stage C"]["scores"].items():
        row = {"outer fold": record["fold"], "model": model_name, **scores}
        rows.append(row)
long_table = pd.DataFrame(rows)
long_table.to_csv(RESULTS / "results_by_fold.csv", index=False)
results_table = long_table.pivot(index="outer fold", columns="model",
                                 values=["Harrell's C", "Uno's C (5 years)"])
results_table.loc["mean"] = results_table.mean()
results_table.loc["SD"] = results_table.iloc[:-1].std()
display(Markdown(f"Outer test scores, {len(fold_records)} of {OUTER_FOLDS} folds."))
results_table.round(3)

In [ ]:
means = results_table.loc["mean", "Harrell's C"]
display(Markdown(f"Mean Harrell's C over the folds: graph model {means['graph model']:.3f}, "
                 f"clinical-only Cox {means['clinical-only Cox']:.3f}, selector as a model "
                 f"{means['selector as a model']:.3f}."))

## Paired comparisons

For each fold the difference between two models scored on the same test part is taken; the mean difference gets a corrected resampled t interval (variance factor 1/5 + 1/4, t with 4 degrees of freedom), which is wide with five values. An interval that includes 0 does not show a difference and does not show equivalence.

In [ ]:
def paired_interval(wide, first, second):
    """Mean paired difference and its corrected resampled t interval over the outer folds."""
    difference = (wide[first] - wide[second]).dropna()
    count = len(difference)
    spread = difference.std(ddof=1) * np.sqrt(1 / count + 1 / (count - 1))
    half_width = stats.t.ppf(0.975, count - 1) * spread
    return {"comparison": f"{first} minus {second}", "mean difference": difference.mean(),
            "lower": difference.mean() - half_width, "upper": difference.mean() + half_width,
            "folds": count}


wide_c = long_table.pivot(index="outer fold", columns="model", values="Harrell's C")
comparisons = [paired_interval(wide_c, "graph model", "clinical-only Cox"),
               paired_interval(wide_c, "graph model", "selector as a model")]
paired = pd.DataFrame(comparisons).set_index("comparison")
paired.to_csv(RESULTS / "paired_comparisons.csv")
paired.round(3)

In [ ]:
first_row = paired.iloc[0]
verdict = "includes 0, so no difference is shown" if first_row["lower"] <= 0 <= first_row["upper"] \
    else "excludes 0"
display(Markdown(f"Graph model minus clinical-only Cox: {first_row['mean difference']:.3f} "
                 f"(interval {first_row['lower']:.3f} to {first_row['upper']:.3f}); the interval "
                 f"{verdict}."))

In [ ]:
figure, axis = plt.subplots(figsize=(7, 4))
for position, model_name in enumerate(long_table["model"].unique()):
    scores = long_table.loc[long_table["model"] == model_name, "Harrell's C"]
    axis.plot([position] * len(scores), scores, "o", label=model_name)
    axis.plot([position - 0.2, position + 0.2], [scores.mean()] * 2, color="black")
axis.set_xticks(range(long_table["model"].nunique()), long_table["model"].unique())
axis.set_ylabel("Harrell's C on the outer test part")
axis.set_xlabel("Model (one dot per outer fold, bar at the mean)")
plt.show()
graph_scores = long_table.loc[long_table["model"] == "graph model", "Harrell's C"]
row_means = long_table.groupby("model")["Harrell's C"].mean()
display(Markdown(f"The graph model runs from {graph_scores.min():.3f} to {graph_scores.max():.3f} "
                 f"across the folds, while the row means lie within "
                 f"{row_means.max() - row_means.min():.3f} of each other."))

## Comparison rows

The comparison rows reuse each fold's split, inner folds and selector setting from its saved file; the fold's recorded scores are never read. Three plain networks (clinical only, genes only, genes plus clinical) and three graph variants (STRING 400, a learnable gene identity at 700, both) each choose their width and dropout on the same inner folds. The selector plus the clinical covariates runs its own stage A on the same paths. The control without edges takes the gene-identity row's genes, width, dropout and seed.

First the graphs themselves: how many STRING edges each fold's gene list has at score 700 and at 400, and the mean number of edges per gene. The same table gives the edges a random list of the same size would have, drawn from the genes the selector could choose in that fold, those passing the expression filter on its outer training part.

In [ ]:
def edge_counts(genes, edge_table):
    """Edges among the given genes and the mean number of edges per gene."""
    kept = edges_among(genes, edge_table)
    return len(kept), round(2 * len(kept) / len(genes), 2)


rows = []
fold_genes = {}
for record in fold_records:
    genes = (RESULTS / f"outer_fold_{record['fold']}_genes.txt").read_text().split()
    fold_genes[record["fold"]] = genes
    edges_700, degree_700 = edge_counts(genes, edges)
    edges_400, degree_400 = edge_counts(genes, denser_edges)
    row = {"outer fold": record["fold"], "genes": len(genes), "edges at 700": edges_700,
           "mean degree at 700": degree_700, "edges at 400": edges_400,
           "mean degree at 400": degree_400}
    rows.append(row)
edge_table = pd.DataFrame(rows).set_index("outer fold")
edge_table.to_csv(RESULTS / "graph_question_edges.csv")

In [ ]:
candidates = {}
expected_rows = []
for record in fold_records:
    fold = record["fold"]
    train = table.iloc[outer[fold - 1][0]]
    assert len(train) == record["patients"]["training"], "the split differs from the saved fold"
    expressed = (train[GENES] >= EXPRESSED_AT).mean() >= EXPRESSED_IN  # the filter of the fold
    candidates[fold] = list(expressed.index[expressed])
    assert set(fold_genes[fold]) <= set(candidates[fold]), "a selected gene is not a candidate"
    list_size, pool_size = len(fold_genes[fold]), len(candidates[fold])
    share = list_size * (list_size - 1) / (pool_size * (pool_size - 1))  # pairs in list / in pool
    row = {"outer fold": fold, "candidate genes": pool_size}
    for label, edge_list in [("700", edges), ("400", denser_edges)]:
        among_candidates = len(edges_among(candidates[fold], edge_list))
        row[f"edges among candidates at {label}"] = among_candidates
        row[f"expected for a random list at {label}"] = round(share * among_candidates, 1)
    expected_rows.append(row)
expected_table = pd.DataFrame(expected_rows).set_index("outer fold")
expected_table.to_csv(RESULTS / "graph_question_expected_edges.csv")
side_by_side = edge_table.join(expected_table)
side_by_side[["genes", "edges at 700", "expected for a random list at 700", "mean degree at 700",
              "edges at 400", "expected for a random list at 400", "mean degree at 400"]]

In [ ]:
ratio_700 = edge_table["edges at 700"] / expected_table["expected for a random list at 700"]
ratio_400 = edge_table["edges at 400"] / expected_table["expected for a random list at 400"]
display(Markdown(f"The selected lists carry {ratio_700.min():.2f} to {ratio_700.max():.2f} times "
                 f"the edges a random list of the same size would carry at 700, and "
                 f"{ratio_400.min():.2f} to {ratio_400.max():.2f} times at 400; a ratio below 1 "
                 "means the selection keeps fewer connected genes than chance."))

A graph the selection cannot empty would add to each selected gene its STRING neighbours at 700 among the same candidates. The table counts the nodes and edges of such a graph with every neighbour and with at most the 5 strongest per gene, and a rough time per fold scaled from the registered run by the number of nodes; nothing is trained here.

In [ ]:
def with_neighbours(genes, candidate_edges, kept=None):
    """The selected genes plus their STRING neighbours among the candidates, strongest first."""
    forward = {"gene_a": "gene", "gene_b": "neighbour"}
    backward = {"gene_b": "gene", "gene_a": "neighbour"}
    both_ways = pd.concat([candidate_edges.rename(columns=forward),
                           candidate_edges.rename(columns=backward)])
    around = both_ways[both_ways["gene"].isin(genes)]
    around = around.sort_values(["gene", "score", "neighbour"], ascending=[True, False, True])
    if kept is not None:
        around = around.groupby("gene").head(kept)
    return sorted(set(genes) | set(around["neighbour"]))


neighbour_rows = []
for record in fold_records:
    genes = fold_genes[record["fold"]]
    candidate_edges = edges_among(candidates[record["fold"]], edges)
    every = with_neighbours(genes, candidate_edges)
    strongest = with_neighbours(genes, candidate_edges, NEIGHBOURS_KEPT)
    strongest_edges = len(edges_among(strongest, candidate_edges))
    minutes = record["minutes"]["stage B"] + record["minutes"]["stage C"]
    row = {"outer fold": record["fold"], "genes": len(genes), "nodes, every neighbour": len(every),
           "edges, every neighbour": len(edges_among(every, candidate_edges)),
           f"nodes, at most {NEIGHBOURS_KEPT} per gene": len(strongest),
           f"edges, at most {NEIGHBOURS_KEPT} per gene": strongest_edges,
           "minutes now, stages B and C": round(minutes, 1),
           f"minutes estimated, at most {NEIGHBOURS_KEPT} per gene":
               round(minutes * len(strongest) / len(genes))}
    neighbour_rows.append(row)
neighbour_table = pd.DataFrame(neighbour_rows).set_index("outer fold")
neighbour_table.to_csv(RESULTS / "graph_question_neighbourhood.csv")
neighbour_table

In [ ]:
kept_nodes = neighbour_table[f"nodes, at most {NEIGHBOURS_KEPT} per gene"]
kept_edges = neighbour_table[f"edges, at most {NEIGHBOURS_KEPT} per gene"]
hours_now = neighbour_table["minutes now, stages B and C"].sum() / 60
hours_kept = neighbour_table[f"minutes estimated, at most {NEIGHBOURS_KEPT} per gene"].sum() / 60
display(Markdown(f"With at most {NEIGHBOURS_KEPT} neighbours per gene the graphs would hold "
                 f"{kept_nodes.min()} to {kept_nodes.max()} genes and {kept_edges.min()} to "
                 f"{kept_edges.max()} edges, and stages B and C would take roughly "
                 f"{hours_kept:.0f} hours against {hours_now:.1f} now, if the time grows with the "
                 "number of nodes."))

### Harrell's C on the outer test parts

The comparison rows that have a saved file for every fold join the registered rows in one table, with the mean and standard deviation over the folds.

In [ ]:
comparison_rows = []
for path in sorted(ROW_DIR.glob("*/outer_fold_*.json")) if ROW_DIR.exists() else []:
    record = json.loads(path.read_text())
    row = {"outer fold": record["fold"], "model": record["row"], "genes": record["genes"],
           **record["scores"]}
    comparison_rows.append(row)
comparison_table = pd.DataFrame(comparison_rows, columns=["outer fold", "model", "genes",
                                                          "Harrell's C", "Uno's C (5 years)"])
complete = comparison_table.groupby("model")["outer fold"].count()
complete_rows = list(complete.index[complete == len(fold_records)])
all_rows = pd.concat([long_table, comparison_table[comparison_table["model"].isin(complete_rows)]])
all_rows.to_csv(RESULTS / "comparison_by_fold.csv", index=False)
question_table = all_rows.pivot(index="outer fold", columns="model",
                                values=["Harrell's C", "Uno's C (5 years)"])
question_table.loc["mean"] = question_table.mean()
question_table.loc["SD"] = question_table.iloc[:-1].std()
display(question_table["Harrell's C"].T.round(3))  # one model per line
harrell_means = question_table.loc["mean", "Harrell's C"]
display(Markdown(f"{len(complete_rows)} comparison rows have all {len(fold_records)} folds; the "
                 f"row means run from {harrell_means.min():.3f} ({harrell_means.idxmin()}) to "
                 f"{harrell_means.max():.3f} ({harrell_means.idxmax()})."))

### Uno's C up to five years

The same rows on Uno's C, which reweights the patient pairs for censoring and is read up to five years.

In [ ]:
display(question_table["Uno's C (5 years)"].T.round(3))  # one model per line
uno_means = question_table.loc["mean", "Uno's C (5 years)"]
display(Markdown(f"On Uno's C the row means run from {uno_means.min():.3f} "
                 f"({uno_means.idxmin()}) to {uno_means.max():.3f} ({uno_means.idxmax()})."))

### Paired differences

Each difference is taken fold by fold on the same test parts, with the corrected resampled t interval of the registered comparison. The four tables answer four questions: does a row beat the clinical-only Cox; do the edges change the score of a graph model against the same model without edges; do gene identity or a denser graph change it; and do the plain networks differ from the graph model and the Cox models.

In [ ]:
PAIRED_GROUPS = {
    "do the edges change the score": [
        ("graph model", "no-graph model"),
        ("graph model, STRING 400", "no-graph model"),
        (IDENTITY_700, "no-graph model, gene identity"),
        ("graph model, STRING 400, gene identity", "no-graph model, gene identity")],
    "do gene identity or a denser graph change it": [
        (IDENTITY_700, "graph model"),
        ("graph model, STRING 400, gene identity", "graph model, STRING 400"),
        ("graph model, STRING 400", "graph model"),
        ("no-graph model, gene identity", "no-graph model")],
    "do the plain networks differ": [
        ("graph model", "gene plus clinical MLP"),
        ("gene plus clinical MLP", "clinical MLP"),
        ("gene MLP", "selector as a model"),
        ("gene plus clinical MLP", "selector plus clinical covariates")]}
against_clinical = []
for name in all_rows["model"].unique():
    if name != "clinical-only Cox":
        against_clinical.append((name, "clinical-only Cox"))
PAIRED_GROUPS["does a row beat the clinical-only Cox"] = against_clinical

In [ ]:
wide_all = all_rows.pivot(index="outer fold", columns="model", values="Harrell's C")
pair_rows = []
for group, pairs in PAIRED_GROUPS.items():
    for first, second in pairs:
        if first in wide_all.columns and second in wide_all.columns:
            result = paired_interval(wide_all, first, second)
            result["question"] = group
            pair_rows.append(result)
question_pairs = pd.DataFrame(pair_rows).set_index("comparison")
question_pairs["includes 0"] = (question_pairs["lower"] <= 0) & (question_pairs["upper"] >= 0)
yes_or_no = {True: "yes", False: "no"}
question_pairs["includes 0"] = question_pairs["includes 0"].map(yes_or_no)
question_pairs.to_csv(RESULTS / "comparison_paired.csv")


def pair_group(group):
    """The paired comparisons that answer one question, rounded for reading."""
    chosen = question_pairs[question_pairs["question"] == group]
    return chosen[["mean difference", "lower", "upper", "includes 0"]].round(3)


def group_sentence(group):
    """How many intervals of one question include 0, and the range of the mean differences."""
    chosen = pair_group(group)
    return (f"{int((chosen['includes 0'] == 'yes').sum())} of the {len(chosen)} intervals include "
            f"0; the mean differences run from {chosen['mean difference'].min():.3f} to "
            f"{chosen['mean difference'].max():.3f}.")

In [ ]:
display(pair_group("does a row beat the clinical-only Cox"))
display(Markdown("Against the clinical-only Cox: "
                 + group_sentence("does a row beat the clinical-only Cox")))

In [ ]:
edge_effects = pair_group("do the edges change the score")
display(edge_effects)
plain_bound = edge_effects.loc["graph model minus no-graph model"]
identity_bound = edge_effects.loc[f"{IDENTITY_700} minus no-graph model, gene identity"]
display(Markdown(f"At score 700 the edges change Harrell's C by {plain_bound['lower']:+.3f} to "
                 f"{plain_bound['upper']:+.3f} without gene identity and by "
                 f"{identity_bound['lower']:+.3f} to {identity_bound['upper']:+.3f} with it. "
                 + group_sentence("do the edges change the score")))

In [ ]:
display(pair_group("do gene identity or a denser graph change it"))
display(Markdown("Gene identity and graph density: "
                 + group_sentence("do gene identity or a denser graph change it")))

In [ ]:
display(pair_group("do the plain networks differ"))
display(Markdown("Plain networks: " + group_sentence("do the plain networks differ")))

In [ ]:
figure, axis = plt.subplots(figsize=(9, 4))
for position, model_name in enumerate(all_rows["model"].unique()):
    scores = all_rows.loc[all_rows["model"] == model_name, "Harrell's C"]
    axis.plot([position] * len(scores), scores, "o")
    axis.plot([position - 0.2, position + 0.2], [scores.mean()] * 2, color="black")
axis.set_xticks(range(all_rows["model"].nunique()), all_rows["model"].unique(), rotation=30,
                ha="right", fontsize=8)
axis.set_ylabel("Harrell's C on the outer test part")
axis.set_xlabel("Row (one dot per outer fold, bar at the mean)")
plt.show()
spans = all_rows.groupby("model")["Harrell's C"].agg(lambda scores: scores.max() - scores.min())
display(Markdown(f"The row means lie between {harrell_means.min():.3f} and "
                 f"{harrell_means.max():.3f}, while each row's folds span {spans.min():.3f} to "
                 f"{spans.max():.3f}."))

## Part 2: the final model

A plain five-fold cross-validation on all patients runs stage A then stage B once more to choose the final settings; the selector refitted on all patients, at the chosen place on their own path, gives the final gene list; the graph model is trained on all patients and frozen. The list need not equal any fold's list, and its size is not fixed in advance.

In [ ]:
final_dir = RESULTS / "final_model"
final_file = final_dir / "model.pt"
if COMPUTE and len(fold_records) == OUTER_FOLDS and not final_file.exists():
    final_dir.mkdir(exist_ok=True)
    started = time.time()
    all_inner = folds(table, INNER_FOLDS)
    setting, grid, per_fold = stage_a(table, all_inner)
    graph_setting, inner_table, few_genes = stage_b(table, all_inner, setting, BASE)
    final_genes, _, penalty = select_genes(table, table, setting)
    x_all, _ = scaled_expression(table, table)
    c_all, _ = clinical_vectors(table, table)
    inputs = model_inputs(BASE, x_all, final_genes, c_all)
    hidden = int(graph_setting["hidden size"])
    dropout = float(graph_setting["dropout"])
    model = train_network(new_model(BASE, inputs, hidden, dropout), inputs, table)
    frozen = {"state": model.state_dict(), "hidden size": hidden, "dropout": dropout,
              "genes": final_genes, "l1 ratio": float(setting["l1 ratio"]),
              "place": int(setting["place"]), "penalty": penalty,
              "clinical levels": {name: sorted(table[name].unique()) for name in CATEGORIES},
              "data sha256": data_hash, "edges sha256": edge_hash}
    torch.save(frozen, final_file)
    final_record = {"stage A": stage_a_record(setting, grid, per_fold),
                    "stage B": stage_b_record(graph_setting, inner_table, few_genes),
                    "genes": len(final_genes), "penalty": penalty,
                    "minutes": round((time.time() - started) / 60, 1),
                    "model sha256": hashlib.sha256(final_file.read_bytes()).hexdigest()}
    (final_dir / "final_record.json").write_text(json.dumps(final_record, indent=1))
    (final_dir / "final_genes.txt").write_text("\n".join(final_genes) + "\n")

In [ ]:
if final_file.exists():
    final_record = json.loads((final_dir / "final_record.json").read_text())
    final_genes = (final_dir / "final_genes.txt").read_text().split()
    rows = []
    for record in fold_records:
        fold_list = set((RESULTS / f"outer_fold_{record['fold']}_genes.txt").read_text().split())
        overlap = len(fold_list & set(final_genes)) / len(fold_list | set(final_genes))
        row = {"outer fold": record["fold"], "genes": len(fold_list),
               "Jaccard overlap with the final list": round(overlap, 2)}
        rows.append(row)
    overlaps = pd.DataFrame(rows).set_index("outer fold")
    assert (overlaps["Jaccard overlap with the final list"] < 1).any(), "no fold list differs"
    final_values = {"L1 ratio": final_record["stage A"]["l1 ratio"],
                    "place on the path": final_record["stage A"]["place"],
                    "penalty": round(final_record["penalty"], 4),
                    "hidden size": final_record["stage B"]["hidden size"],
                    "dropout": final_record["stage B"]["dropout"],
                    "final genes": final_record["genes"],
                    "model file sha256": final_record["model sha256"][:16] + "...",
                    "minutes": final_record["minutes"]}
    final_table = pd.Series(final_values, name="final model").to_frame()
    final_table.to_csv(RESULTS / "final_settings.csv")
    overlaps.to_csv(RESULTS / "final_overlaps.csv")
    display(final_table, overlaps)
    jaccard = overlaps["Jaccard overlap with the final list"]
    display(Markdown(f"The final list of {final_record['genes']} genes has a Jaccard overlap of "
                     f"{jaccard.min():.2f} to {jaccard.max():.2f} with the five fold lists, the "
                     "shared genes divided by the genes in either list."))
else:
    display(Markdown("The final model is not yet trained."))

The METABRIC cell below is switched off; the frozen model scores the external cohort once. The alignment is set with the supervisors before the first and only run; this cell is rewritten then and does not run until then.

In [ ]:
if SCORE_METABRIC and final_file.exists():
    frozen = torch.load(final_file, weights_only=False)
    metabric = pd.read_parquet("../data/processed/metabric.parquet")
    x_metabric, _ = scaled_expression(metabric, metabric)
    c_metabric, _ = clinical_vectors(metabric, metabric)
    inputs = model_inputs(BASE, x_metabric, frozen["genes"], c_metabric)
    model = new_model(BASE, inputs, frozen["hidden size"], frozen["dropout"])
    model.load_state_dict(frozen["state"])
    risk = log_hazard(model, inputs)
    external = {"Harrell's C": float(harrell(metabric, risk)),
                "Uno's C (5 years)": float(uno(table, metabric, risk)), "patients": len(metabric)}
    (RESULTS / "metabric_score.json").write_text(json.dumps(external, indent=1))
    display(pd.Series(external, name="METABRIC").to_frame())
else:
    display(Markdown("METABRIC is not scored in this version; the switch in the settings is set "
                     "to no."))

## Summary file and run record

The summary file holds the numbers the meeting pages use beside the tables above. The run record names the git commit, the package versions, the seeds, the input hashes and the minutes per stage.

In [ ]:
summary = {"outer folds done": len(fold_records)}
for model_name in results_table["Harrell's C"].columns:
    summary[f"{model_name} mean C"] = round(float(means[model_name]), 3)
    spread = results_table.loc["SD", ("Harrell's C", model_name)]
    summary[f"{model_name} SD C"] = round(float(spread), 3)
    mean_uno = results_table.loc["mean", ("Uno's C (5 years)", model_name)]
    summary[f"{model_name} mean Uno C"] = round(float(mean_uno), 3)
for name, row in paired.iterrows():
    short = "graph minus clinical" if "clinical" in name else "graph minus selector"
    summary[f"{short} mean"] = round(float(row["mean difference"]), 3)
    summary[f"{short} lower"] = round(float(row["lower"]), 3)
    summary[f"{short} upper"] = round(float(row["upper"]), 3)
for model_name in complete_rows:
    mean_c = question_table.loc["mean", ("Harrell's C", model_name)]
    spread_c = question_table.loc["SD", ("Harrell's C", model_name)]
    mean_uno = question_table.loc["mean", ("Uno's C (5 years)", model_name)]
    summary[f"{model_name} mean C"] = round(float(mean_c), 3)
    summary[f"{model_name} SD C"] = round(float(spread_c), 3)
    summary[f"{model_name} mean Uno C"] = round(float(mean_uno), 3)
for name, row in question_pairs.iterrows():
    summary[f"{name} mean"] = round(float(row["mean difference"]), 3)
    summary[f"{name} lower"] = round(float(row["lower"]), 3)
    summary[f"{name} upper"] = round(float(row["upper"]), 3)
summary["genes selected, smallest"] = int(stage_a_table["genes selected"].min())
summary["genes selected, largest"] = int(stage_a_table["genes selected"].max())
summary["final model trained"] = "yes" if final_file.exists() else "no"
if final_file.exists():
    summary["final genes"] = final_record["genes"]
pd.Series(summary, name="value").to_csv(RESULTS / "summary.csv")
display(Markdown(f"{len(summary)} numbers are saved in the summary file for the meeting pages."))

In [ ]:
def git(*arguments):
    """The output of one git command, for the run record."""
    command = ["git", *arguments]
    return subprocess.run(command, capture_output=True, text=True, check=True).stdout.strip()


packages = {}
for name in ["numpy", "pandas", "scikit-learn", "scikit-survival", "torch", "torch_geometric",
             "torchsurv"]:
    packages[name] = version(name)
minutes = {}
for record in fold_records:
    stage_minutes = record["minutes"]
    minutes[f"outer fold {record['fold']}"] = (f"stage A {stage_minutes['stage A']}, "
                                               f"stage B {stage_minutes['stage B']}, "
                                               f"stage C {stage_minutes['stage C']}")
minutes["comparison rows"] = "the minutes of each row and fold are in its file"
run_record = {"git commit": git("rev-parse", "HEAD"),
              "tracked files changed since that commit":
                  git("status", "--porcelain", "--untracked-files=no").splitlines(),
              "mode": "computed" if COMPUTE else "read the saved results",
              "python": sys.version.split()[0], "packages": packages,
              "seeds": {"folds": FOLD_SEED, "torch": TORCH_SEED},
              "cohort table (sha256)": data_hash, "edge list (sha256)": edge_hash,
              "minutes": minutes}
record_name = "compute_record.json" if COMPUTE else "run_record.json"
(RESULTS / record_name).write_text(json.dumps(run_record, indent=1))
print(json.dumps(run_record, indent=1))